# Generic Parameter Robustness Heatmaps

This notebook scans GeoPAS BBOB result directories under `results/bbob_by_deepela/results/bbob/<protocol>/`, filters them by a user-specified `USER_FILTERS` dictionary, parses the aggregate `res_*.csv` summaries, and plots heatmaps over any two user-chosen run parameters.

The current run signature is `scale..._head2scale..._sigmoidlogs..._dual..._head2lw..._head2sw..._priorscale..._lamprior..._tailscale...`, so any parameter you are not plotting should usually stay fixed in `USER_FILTERS`. If one of the plotted parameters appears in `USER_FILTERS`, leave it as `None`.

Suggested workflow:
1. Set `PLOT_Y_PARAM`, `PLOT_X_PARAM`, and `USER_FILTERS` in Cell 2.
2. Run Cell 3 to inspect the matching fixed-parameter slices and tighten the remaining filters until the desired slice is isolated.
3. Run Cell 4 to build `df_grid` and confirm how many unique values were loaded for the two plotted parameters.
4. Run Cell 5 once per kernel session to load the plotting helpers.
5. Run Cells 6 and 7 to generate the summary and bounded-performance heatmaps.

In [ ]:
from __future__ import annotations

import csv
import math
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "GeoPAS_v1").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")

PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", os.environ.get("GEOPAS_PROJECT_ROOT", str(find_project_root(Path.cwd().resolve()))))
).resolve()
DEFAULT_RESULTS_ROOT = PROJECT_ROOT / "results" / "bbob_by_deepela" / "results" / "bbob"
RESULTS_ROOT = Path(os.environ.get("RESULTS_ROOT", str(DEFAULT_RESULTS_ROOT))).resolve()

SAVE_DIR = PROJECT_ROOT / "GeoPAS_v1" / "analysis_outputs" / "parameter_heatmaps"

RUN_PARAMETER_COLUMNS = (
    "target_scale",
    "head_2_target_scale",
    "prior_scale",
    "sigmoid_log_s",
    "tail_scale",
    "lam_prior",
    "dual_head",
    "head_2_loss_weight",
    "head_2_score_weight",
    "res",
    "k_views",
)

PLOT_Y_PARAM = "sigmoid_log_s"
PLOT_X_PARAM = "lam_prior"

USER_FILTERS = {
    "protocol": "lio",
    "target_scale": "log_power",
    "head_2_target_scale": "norm",
    "prior_scale": "log_power",
    "sigmoid_log_s": 1.0,
    "tail_scale": 1.0,
    "lam_prior": None,
    "dual_head": 0,
    "head_2_loss_weight": 0.0,
    "head_2_score_weight": 0.0,
    "res": 8,
    "k_views": 32,
    "seed": None,
}

NUMBER_PATTERN = r"[+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?"
RUN_DIR_RE = re.compile(
    rf"scale(?P<target_scale>[A-Za-z0-9_]+)"
    rf"_head2scale(?P<head_2_target_scale>[A-Za-z0-9_]+)"
    rf"_sigmoidlogs(?P<sigmoid_log_s>{NUMBER_PATTERN})"
    rf"_dual(?P<dual_head>\d+)"
    rf"_head2lw(?P<head_2_loss_weight>{NUMBER_PATTERN})"
    rf"_head2sw(?P<head_2_score_weight>{NUMBER_PATTERN})"
    rf"_priorscale(?P<prior_scale>[A-Za-z0-9_]+)"
    rf"_lamprior(?P<lam_prior>{NUMBER_PATTERN})"
    rf"_tailscale(?P<tail_scale>{NUMBER_PATTERN})"
    rf"_res_(?P<res>\d+)"
    rf"_k_views_(?P<k_views>\d+)"
)

SUMMARY_METRICS = ("Mean", "Median", "P90")
SUMMARY_SECTIONS = ("AS", "VBS", "SBS", "Gap_Closure")
SINGLE_TABLE_METRICS = {
    "Accuracies": "accuracy",
    "F1": "f1",
}
STOP_LABELS = set(SUMMARY_METRICS) | set(SUMMARY_SECTIONS) | set(SINGLE_TABLE_METRICS) | {"Pick_Rate", "VBS_Pick_Rate"}

class ParseError(RuntimeError):
    pass

def _skip_blank_lines(lines: List[str], start_idx: int) -> int:
    i = start_idx
    while i < len(lines) and not lines[i].strip():
        i += 1
    return i

def _parse_csv_row(line: str) -> List[str]:
    return next(csv.reader([line]))

def _parse_all_all_value(lines: List[str], start_idx: int, label: str) -> Tuple[float, int]:
    i = _skip_blank_lines(lines, start_idx)
    if i >= len(lines) or not lines[i].lstrip().startswith("Problem Group"):
        got = lines[i].rstrip() if i < len(lines) else "<EOF>"
        raise ParseError(f"Expected 'Problem Group,...' after {label}, got: {got}")
    i += 1

    all_row: Optional[List[str]] = None
    while i < len(lines):
        stripped = lines[i].strip()
        if not stripped or stripped in STOP_LABELS:
            break
        row = [token.strip() for token in _parse_csv_row(lines[i])]
        if row and row[0].lower() == "all":
            all_row = row
        i += 1

    if all_row is None:
        raise ParseError(f"Did not find an 'all' row for {label}")

    value_text = all_row[-1]
    try:
        return float(value_text), i
    except ValueError as exc:
        raise ParseError(f"Could not parse {label} all/all value {value_text!r}") from exc

def parse_run_dir_name(run_dir_name: str) -> Dict[str, object]:
    match = RUN_DIR_RE.fullmatch(run_dir_name)
    if not match:
        raise ParseError(f"Unrecognised run directory name: {run_dir_name}")
    groups = match.groupdict()
    return {
        "target_scale": groups["target_scale"],
        "head_2_target_scale": groups["head_2_target_scale"],
        "prior_scale": groups["prior_scale"],
        "sigmoid_log_s": float(groups["sigmoid_log_s"]),
        "tail_scale": float(groups["tail_scale"]),
        "lam_prior": float(groups["lam_prior"]),
        "dual_head": int(groups["dual_head"]),
        "head_2_loss_weight": float(groups["head_2_loss_weight"]),
        "head_2_score_weight": float(groups["head_2_score_weight"]),
        "res": int(groups["res"]),
        "k_views": int(groups["k_views"]),
    }

def parse_result_summary_csv(csv_path: str | Path) -> Dict[str, float]:
    csv_path = Path(csv_path)
    lines = csv_path.read_text(encoding="utf-8", errors="replace").splitlines()

    metrics: Dict[str, float] = {}
    i = 0
    while i < len(lines):
        i = _skip_blank_lines(lines, i)
        if i >= len(lines):
            break

        label = lines[i].strip()
        if label in SUMMARY_METRICS:
            metric_key = label.lower()
            i += 1
            section_values: Dict[str, float] = {}
            for section in SUMMARY_SECTIONS:
                i = _skip_blank_lines(lines, i)
                if i >= len(lines) or lines[i].strip() != section:
                    got = lines[i].strip() if i < len(lines) else "<EOF>"
                    raise ParseError(f"Expected section {section!r} inside {label}, got {got!r}")
                i += 1
                section_values[section], i = _parse_all_all_value(lines, i, f"{label}/{section}")

            metrics[metric_key] = section_values["AS"]
            metrics[f"sbs_{metric_key}"] = section_values["SBS"]
            metrics[f"vbs_{metric_key}"] = section_values["VBS"]
            metrics[f"gap_closure_{metric_key}"] = section_values["Gap_Closure"]
            continue

        if label in SINGLE_TABLE_METRICS:
            metric_key = SINGLE_TABLE_METRICS[label]
            i += 1
            metrics[metric_key], i = _parse_all_all_value(lines, i, label)
            continue

        if label in {"Pick_Rate", "VBS_Pick_Rate"}:
            break

        i += 1

    required_metrics = {"mean", "median", "p90"}
    missing = sorted(required_metrics - set(metrics))
    if missing:
        raise ParseError(f"Missing required metrics {missing} in {csv_path}")

    return metrics

def _matches_filter(actual: object, expected: object) -> bool:
    if expected is None:
        return True
    if isinstance(expected, (list, tuple, set, frozenset)):
        return any(_matches_filter(actual, candidate) for candidate in expected)
    if isinstance(expected, float):
        return math.isclose(float(actual), expected, rel_tol=0.0, abs_tol=1e-9)
    return actual == expected

def _filter_dataframe_by_user_filters(df: pd.DataFrame, filters: Dict[str, object]) -> pd.DataFrame:
    filtered = df.copy()
    for key, expected in filters.items():
        if key not in filtered.columns or expected is None:
            continue
        filtered = filtered[filtered[key].map(lambda actual: _matches_filter(actual, expected))]
    return filtered.reset_index(drop=True)

In [2]:
def find_result_summary_csv(run_dir: Path, seed: Optional[int] = None) -> Optional[Path]:
    base_dir = run_dir if seed is None else run_dir / f"seed{int(seed)}"
    if not base_dir.exists():
        return None
    candidates = sorted(base_dir.glob("res_*.csv"))
    return candidates[0] if len(candidates) == 1 else None


def list_available_parameter_slices(
    filters: Dict[str, object],
    *,
    varying_cols: Tuple[str, str],
    results_root: Path = RESULTS_ROOT,
) -> pd.DataFrame:
    protocol = str(filters["protocol"])
    protocol_dir = results_root / protocol
    if not protocol_dir.exists():
        return pd.DataFrame()

    rows = []
    for run_dir in sorted(protocol_dir.iterdir()):
        if not run_dir.is_dir():
            continue
        if find_result_summary_csv(run_dir, seed=filters.get("seed")) is None:
            continue
        try:
            params = parse_run_dir_name(run_dir.name)
        except ParseError:
            continue
        rows.append(params)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    slice_cols = [column for column in RUN_PARAMETER_COLUMNS if column not in set(varying_cols)]
    available = (
        df.groupby(slice_cols, dropna=False)
        .size()
        .rename("n_plot_parameter_pairs")
        .reset_index()
        .sort_values(slice_cols)
        .reset_index(drop=True)
    )
    return _filter_dataframe_by_user_filters(available, filters)


invalid_plot_params = [column for column in (PLOT_Y_PARAM, PLOT_X_PARAM) if column not in RUN_PARAMETER_COLUMNS]
if invalid_plot_params:
    print(f"Unknown plot parameters: {invalid_plot_params}")
    print(f"Choose from: {', '.join(RUN_PARAMETER_COLUMNS)}")
    available_slices = pd.DataFrame()
elif PLOT_X_PARAM == PLOT_Y_PARAM:
    print("Choose different values for PLOT_X_PARAM and PLOT_Y_PARAM.")
    available_slices = pd.DataFrame()
else:
    available_slices = list_available_parameter_slices(
        USER_FILTERS,
        varying_cols=(PLOT_Y_PARAM, PLOT_X_PARAM),
    )
    if available_slices.empty:
        print("No matching fixed-parameter slices were found for the current USER_FILTERS.")
    else:
        total_pairs = int(available_slices["n_plot_parameter_pairs"].sum())
        print(
            f"Found {len(available_slices)} fixed-parameter slice(s), covering {total_pairs} ({PLOT_Y_PARAM}, {PLOT_X_PARAM}) pair(s)."
        )

available_slices

Found 1 fixed-parameter slice(s), covering 154 (lam_prior, sigmoid_log_s) pair(s).


,target_scale,head_2_target_scale,prior_scale,tail_scale,dual_head,head_2_loss_weight,head_2_score_weight,res,k_views,n_plot_parameter_pairs
0,log_power,norm,log_power,1.0,0,0.0,0.0,8,32,154


In [3]:
def collect_parameter_grid_results(
    filters: Dict[str, object],
    *,
    x_col: str,
    y_col: str,
    results_root: Path = RESULTS_ROOT,
) -> pd.DataFrame:
    protocol = str(filters["protocol"])
    protocol_dir = results_root / protocol
    if not protocol_dir.exists():
        return pd.DataFrame()

    fixed_filter_keys = {
        key: value
        for key, value in filters.items()
        if key not in {"protocol", "seed", x_col, y_col}
    }

    rows = []
    for run_dir in sorted(protocol_dir.iterdir()):
        if not run_dir.is_dir():
            continue
        summary_csv = find_result_summary_csv(run_dir, seed=filters.get("seed"))
        if summary_csv is None:
            continue
        try:
            params = parse_run_dir_name(run_dir.name)
        except ParseError:
            continue

        if any(key not in params or not _matches_filter(params[key], expected) for key, expected in fixed_filter_keys.items()):
            continue

        metrics = parse_result_summary_csv(summary_csv)
        rows.append({"protocol": protocol, "csv_path": str(summary_csv), **params, **metrics})

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values([y_col, x_col]).reset_index(drop=True)
    duplicate_points = df.duplicated(subset=[x_col, y_col], keep=False)
    if duplicate_points.any():
        print("Duplicate plot-parameter pairs were found for the current USER_FILTERS.")
        display(df.loc[duplicate_points, [y_col, x_col]].drop_duplicates().reset_index(drop=True))
        return pd.DataFrame()

    df["budget"] = df["res"] ** 2 * df["k_views"]
    return df


df_grid = None
if invalid_plot_params:
    print(f"Unknown plot parameters: {invalid_plot_params}")
elif PLOT_X_PARAM == PLOT_Y_PARAM:
    print("Choose different values for PLOT_X_PARAM and PLOT_Y_PARAM.")
elif USER_FILTERS.get(PLOT_X_PARAM) is not None or USER_FILTERS.get(PLOT_Y_PARAM) is not None:
    print(f"Set USER_FILTERS[{PLOT_X_PARAM!r}] and USER_FILTERS[{PLOT_Y_PARAM!r}] to None, then rerun this cell.")
else:
    df_grid = collect_parameter_grid_results(
        USER_FILTERS,
        x_col=PLOT_X_PARAM,
        y_col=PLOT_Y_PARAM,
    )
    if df_grid.empty:
        print("No matching runs were found. Adjust USER_FILTERS after checking available_slices, then rerun this cell.")
        df_grid = None
    else:
        print(
            "Loaded "
            f"{len(df_grid)} runs across "
            f"{df_grid[PLOT_Y_PARAM].nunique()} {PLOT_Y_PARAM} values x "
            f"{df_grid[PLOT_X_PARAM].nunique()} {PLOT_X_PARAM} values."
        )
        display_columns = [
            column
            for column in [PLOT_Y_PARAM, PLOT_X_PARAM, "mean", "median", "p90", "accuracy", "f1"]
            if column in df_grid.columns
        ]
        df_grid[display_columns]

Set USER_FILTERS['sigmoid_log_s'] and USER_FILTERS['lam_prior'] to None, then rerun this cell.


In [4]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

PAPER_BG = "#f9f8f5"
AXIS_BG = "#ffffff"
GRID_COLOR = "#d8cec0"
TEXT_COLOR = "#2d2622"
MISSING_COLOR = "#efe7dc"
HEATMAP_CMAP = LinearSegmentedColormap.from_list(
    "as_bbo_heatmap",
    ["#143d4a", "#336c7a", "#7aa7a2", "#d7d3b1", "#f3eadf"],
)
HEATMAP_CMAP.set_bad(MISSING_COLOR)

PARAM_LABELS = {
    "target_scale": "Target scale",
    "head_2_target_scale": "Head-2 target scale",
    "prior_scale": "Prior scale",
    "sigmoid_log_s": "Sigmoid-log s",
    "tail_scale": "Tail scale",
    "lam_prior": "Prior weight",
    "dual_head": "Dual head",
    "head_2_loss_weight": "Head-2 loss weight",
    "head_2_score_weight": "Head-2 score weight",
    "res": "Resolution",
    "k_views": "K views",
}

mpl.rcParams.update(
    {
        "figure.facecolor": PAPER_BG,
        "axes.facecolor": AXIS_BG,
        "axes.edgecolor": AXIS_BG,
        "axes.labelcolor": TEXT_COLOR,
        "axes.titlecolor": TEXT_COLOR,
        "xtick.color": TEXT_COLOR,
        "ytick.color": TEXT_COLOR,
        "text.color": TEXT_COLOR,
        "font.family": "DejaVu Serif",
        "axes.titlesize": 18,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "figure.dpi": 140,
        "savefig.dpi": 240,
        "savefig.facecolor": PAPER_BG,
        "savefig.bbox": "tight",
    }
)

def _pretty_name(name: str) -> str:
    return PARAM_LABELS.get(name, name.replace("_", " ").title())

def _format_axis_value(value: object) -> str:
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return str(value)
    return f"{int(numeric)}" if numeric.is_integer() else f"{numeric:g}"

def _format_cell_value(value: float) -> str:
    if value == 0:
        return "0"

    magnitude = abs(value)
    order = int(np.floor(np.log10(magnitude)))
    decimals = min(3, max(0, 3 - order))
    rounded = round(value, decimals)

    if rounded == 0:
        return "0"
    if decimals == 0:
        return f"{rounded:.0f}"

    return f"{rounded:.{decimals}f}".rstrip("0").rstrip(".")

def _text_color_from_tile(im: mpl.image.AxesImage, value: float) -> str:
    normed = im.norm(value)
    if np.ma.is_masked(normed):
        return TEXT_COLOR

    normed_array = np.asarray(normed)
    if normed_array.size == 0 or not np.isfinite(normed_array).all():
        return TEXT_COLOR

    rgba = np.asarray(im.cmap(float(normed_array.reshape(-1)[0])))
    if rgba.ndim > 1:
        rgba = rgba.reshape(-1, rgba.shape[-1])[0]
    if rgba.size < 4:
        return TEXT_COLOR

    red, green, blue, _ = rgba[:4]
    luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
    return AXIS_BG if luminance < 0.6 else TEXT_COLOR

def plot_parameter_heatmap(
    df: pd.DataFrame,
    value_col: str,
    title: str,
    *,
    x_col: str,
    y_col: str,
    x_label: Optional[str] = None,
    y_label: Optional[str] = None,
    output_path: Optional[str | Path] = None,
    cbar_range: Optional[Tuple[float, float]] = None,
    cmap: Optional[mpl.colors.Colormap] = None,
    figsize: Optional[Tuple[float, float]] = None,
    log_scale: bool = False,
) -> None:
    pivot = df.pivot_table(index=y_col, columns=x_col, values=value_col, aggfunc="mean")
    pivot = pivot.sort_index().sort_index(axis=1)
    data = np.ma.masked_invalid(pivot.values.astype(float))

    auto_figsize = figsize or (
        0.68 * len(pivot.columns) + 1,
        0.68 * len(pivot.index) + 1,
    )

    fig, ax = plt.subplots(figsize=auto_figsize)

    norm = None
    if log_scale:
        finite_positive = pivot.values[np.isfinite(pivot.values) & (pivot.values > 0)]
        if finite_positive.size and not np.allclose(finite_positive.min(), finite_positive.max()):
            norm = mpl.colors.LogNorm(vmin=float(finite_positive.min()), vmax=float(finite_positive.max()))

    im = ax.imshow(
        data,
        aspect="auto",
        origin="lower",
        cmap=cmap or HEATMAP_CMAP,
        interpolation="nearest",
        norm=norm,
    )

    if cbar_range is not None:
        im.set_clim(*cbar_range)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([_format_axis_value(v) for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([_format_axis_value(v) for v in pivot.index])
    ax.set_xlabel(x_label or _pretty_name(x_col))
    ax.set_ylabel(y_label or _pretty_name(y_col))

    ax.set_xticks(np.arange(-0.5, len(pivot.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(pivot.index), 1), minor=True)
    ax.grid(which="minor", color=GRID_COLOR, linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    for y in range(pivot.shape[0]):
        for x in range(pivot.shape[1]):
            value = pivot.values[y, x]
            if np.isnan(value):
                continue
            ax.text(
                x,
                y,
                _format_cell_value(float(value)),
                ha="center",
                va="center",
                color=_text_color_from_tile(im, float(value)),
                fontsize=11,
                fontweight="semibold",
            )

    protocol_label = str(df["protocol"].iloc[0]).upper()
    target_scale_label = str(df["target_scale"].iloc[0])
    lam_prior_label = fr"$\alpha$={_format_axis_value(df['lam_prior'].iloc[0])}" if "lam_prior" in df.columns else None
    ax.set_title(
        # f"{title} ({protocol_label}, {target_scale_label})",
        # f"{title} ({protocol_label}, {lam_prior_label})",
        f"{title} ({protocol_label})",
        y=1.02,
        fontsize=15,
        fontweight="semibold",
    )

    fig.tight_layout(rect=(0, 0, 1, 0.95))
    if output_path is not None:
        output_dir = Path(output_path)
        output_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_dir / f"{protocol_label.lower()}_{target_scale_label}_{value_col}_{y_col}_vs_{x_col}_heatmap.svg")
    plt.show()

In [5]:
SUMMARY_PLOTS = [
    ("mean", "Mean"),
    ("median", "Median"),
    ("p90", "P90"),
]

if df_grid is None:
    print("Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.")
else:
    for column, title in SUMMARY_PLOTS:
        plot_title = f"{title} (log-scale colour)" if column in {"mean", "p90"} else title
        plot_parameter_heatmap(
            df_grid,
            column,
            plot_title,
            x_col=PLOT_X_PARAM,
            y_col=PLOT_Y_PARAM,
            output_path=SAVE_DIR,
            log_scale=column in {"mean", "p90"},
        )

Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.


In [6]:
BOUNDED_PLOTS = [
    ("accuracy", "Accuracy"),
    ("f1", "F1"),
]

if df_grid is None:
    print("Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.")
else:
    missing_cols = [column for column, _ in BOUNDED_PLOTS if column not in df_grid.columns]
    if missing_cols:
        print(f"Skipping missing columns: {missing_cols}")

    for column, title in BOUNDED_PLOTS:
        if column not in df_grid.columns:
            continue
        # if column == "accuracy":
        #     plot_parameter_heatmap(
        #         df_grid,
        #         column,
        #         title,
        #         x_col=PLOT_X_PARAM,
        #         y_col=PLOT_Y_PARAM,
        #     )
        # else:
        #     plot_parameter_heatmap(
        #         df_grid,
        #         column,
        #         title,
        #         x_col=PLOT_X_PARAM,
        #         y_col=PLOT_Y_PARAM,
        #         output_path=SAVE_DIR,
        #     )
        
        plot_parameter_heatmap(
            df_grid,
            column,
            title,
            x_col=PLOT_X_PARAM,
            y_col=PLOT_Y_PARAM,
            output_path=SAVE_DIR,
        )

Adjust PLOT_X_PARAM, PLOT_Y_PARAM, and USER_FILTERS, rerun Cell 4, then rerun this cell.
